# Demo - Tokenization: BPE and WordPiece
**Day 2 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-tokenization.ipynb)

**Goal:** Show that models see text as a sequence of integer token IDs, and
that subword tokenization splits rare words into known pieces.

We tokenize the same sentences two ways: GPT-style BPE (`tiktoken`) and
BERT-style WordPiece (`transformers`). We print the tokens, the integer IDs,
and the token count so you can see why token count, not word count, is the
real budget.

No API key needed. `tiktoken` runs offline; the BERT tokenizer downloads a
small vocab file once, then caches.

In [1]:
# Scoped install for this demo (safe to re-run; tiktoken is preinstalled on Colab).
%pip install -q tiktoken transformers

In [2]:
import tiktoken
from transformers import AutoTokenizer

## 1. The sentences we will tokenize
Four sentences: a plain one, one about the topic itself, a rare medical term,
and a line of code. Watch how the rare word and the code split into pieces.

In [3]:
SENTENCES = [
    "The cat sat on the mat.",
    "Tokenization splits rare words into subwords.",
    "acetaminophen",
    "def get_weather(city): return api.fetch(city)",
]
for s in SENTENCES:
    print(s)

The cat sat on the mat.
Tokenization splits rare words into subwords.
acetaminophen
def get_weather(city): return api.fetch(city)


## 2. BPE (GPT family, via tiktoken)
`cl100k_base` is the encoding used by GPT-3.5/GPT-4 family models. It runs
fully offline, no download required.

In [4]:
def demo_bpe():
    """GPT-style Byte Pair Encoding with tiktoken (fully offline)."""
    print("1. BPE (GPT family, via tiktoken)")
    encoder = tiktoken.get_encoding("cl100k_base")
    for text in SENTENCES:
        ids = encoder.encode(text)
        pieces = [encoder.decode([i]) for i in ids]
        print(f"\n  text   : {text!r}")
        print(f"  tokens : {pieces}")
        print(f"  ids    : {ids}")
        print(f"  count  : {len(ids)} tokens for {len(text)} characters")


demo_bpe()

1. BPE (GPT family, via tiktoken)

  text   : 'The cat sat on the mat.'
  tokens : ['The', ' cat', ' sat', ' on', ' the', ' mat', '.']
  ids    : [791, 8415, 7731, 389, 279, 5634, 13]
  count  : 7 tokens for 23 characters

  text   : 'Tokenization splits rare words into subwords.'
  tokens : ['Token', 'ization', ' splits', ' rare', ' words', ' into', ' sub', 'words', '.']
  ids    : [3404, 2065, 41567, 9024, 4339, 1139, 1207, 5880, 13]
  count  : 9 tokens for 45 characters

  text   : 'acetaminophen'
  tokens : ['acet', 'amin', 'oph', 'en']
  ids    : [68323, 8778, 5237, 268]
  count  : 4 tokens for 13 characters

  text   : 'def get_weather(city): return api.fetch(city)'
  tokens : ['def', ' get', '_weather', '(city', '):', ' return', ' api', '.fetch', '(city', ')']
  ids    : [755, 636, 70464, 44602, 1680, 471, 6464, 16512, 44602, 8]
  count  : 10 tokens for 45 characters


## 3. WordPiece (BERT family, via transformers)
`bert-base-uncased` downloads a small vocab file once, then caches. A `##`
prefix marks a piece that attaches to the previous token.

In [5]:
def demo_wordpiece():
    """BERT-style WordPiece with the transformers tokenizer."""
    print("2. WordPiece (BERT family, via transformers)")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    for text in SENTENCES:
        tokens = tokenizer.tokenize(text)
        ids = tokenizer.convert_tokens_to_ids(tokens)
        print(f"\n  text   : {text!r}")
        print(f"  tokens : {tokens}")
        print(f"  ids    : {ids}")
        print(f"  count  : {len(tokens)} tokens")
    print("\n  Note: '##' marks a piece that attaches to the previous token.")


demo_wordpiece()

2. WordPiece (BERT family, via transformers)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


  text   : 'The cat sat on the mat.'
  tokens : ['the', 'cat', 'sat', 'on', 'the', 'mat', '.']
  ids    : [1996, 4937, 2938, 2006, 1996, 13523, 1012]
  count  : 7 tokens

  text   : 'Tokenization splits rare words into subwords.'
  tokens : ['token', '##ization', 'splits', 'rare', 'words', 'into', 'sub', '##words', '.']
  ids    : [19204, 3989, 19584, 4678, 2616, 2046, 4942, 22104, 1012]
  count  : 9 tokens

  text   : 'acetaminophen'
  tokens : ['ace', '##tam', '##ino', '##ph', '##en']
  ids    : [9078, 15464, 5740, 8458, 2368]
  count  : 5 tokens

  text   : 'def get_weather(city): return api.fetch(city)'
  tokens : ['def', 'get', '_', 'weather', '(', 'city', ')', ':', 'return', 'api', '.', 'fetch', '(', 'city', ')']
  ids    : [13366, 2131, 1035, 4633, 1006, 2103, 1007, 1024, 2709, 17928, 1012, 18584, 1006, 2103, 1007]
  count  : 15 tokens

  Note: '##' marks a piece that attaches to the previous token.


## Expected output
- Common words (`the`, `cat`, `sat`) stay whole tokens in both tokenizers.
- `acetaminophen` splits into several subword pieces in both tokenizers,
  since it is a rare word not in either vocabulary as a single token.
- The code line splits on punctuation and identifier boundaries.
- Token count is always higher than word count for the rare word and code
  examples, but close to word count for the plain sentence.

**Takeaway:** models count tokens, not words. Rare words cost more tokens.